# Task 1.2 — Key Assumptions
**Paper:** Poisoning Attacks against Support Vector Machines  
**Authors:** Battista Biggio, Blaine Nelson, Pavel Laskov (ICML 2012)  
**Roll Number:** 230110  
**Random Seed:** N/A (no code in this notebook)

---
## Assumption 1: White-Box Knowledge of the Training Data, Learning Algorithm, and Kernel

**Assumption:**  
The attacker has complete knowledge of the target SVM's training dataset $D_{tr}$, the learning algorithm (SVM), the kernel function used (e.g., linear, polynomial, or RBF), and all hyperparameters including the regularisation parameter $C$ and kernel parameter $\gamma$. The attacker can also draw samples from the same underlying data distribution to construct a validation set $D_{val}$.

**Why the method needs it:**  
The entire gradient computation in Equation (10) requires the attacker to compute the kernel matrix $Q$, the SVM dual variables $\alpha$, and the margin conditions $g_k$ — all of which depend on knowing $D_{tr}$, the kernel, and its parameters. Without this knowledge, the attacker cannot compute $\partial L / \partial u$ and therefore cannot determine the optimal direction to perturb the attack point $x_c$. The adiabatic update condition (Eqs. 4–5) also requires knowing which training points are in sets $S$, $E$, and $R$, which is only possible with full knowledge of the SVM solution.

**Violation scenario:**  
In a real-world deployment such as a cloud-based malware classifier or a proprietary spam filter, the defender's training data, kernel choice, and hyperparameters are typically private. An attacker could at best use a surrogate dataset drawn from a similar distribution and guess the kernel family, making the computed gradient a noisy approximation. This would reduce the attack to a *grey-box* or *black-box* scenario where the gradient-based method may converge to suboptimal local maxima or fail to increase test error significantly.

**Paper reference:**  
Section 2, paragraph 1: *"we assume that the attacker knows the learning algorithm and can draw data from the underlying data distribution. Further, we assume that our attacker knows the training data used by the learner."* The authors explicitly acknowledge this is *"generally, an unrealistic assumption"* but justify it as a worst-case analysis.

---
## Assumption 2: The Attacker Controls the Label of the Injected Point

**Assumption:**  
The attacker can inject a training point $(x_c, y_c)$ with an arbitrary but fixed label $y_c$ of their choice, and the learning system will trust and use this label during retraining. The label is not validated or corrected by any external oracle.

**Why the method needs it:**  
The attack begins by flipping the label of a point from the *attacked class* to the *attacking class* (Algorithm 1, initialisation). The gradient derivation in Eq. (10) treats $y_c$ as fixed and optimises only over the feature vector $x_c$. If the attacker cannot control $y_c$, the initial mislabelling that seeds the entire attack chain — placing a point from one class into the other class's training distribution — cannot be achieved. Without label control, the attack reduces to crafting an outlier within the correct class, which has a far weaker effect on the decision boundary.

**Violation scenario:**  
Consider a spam filter where labels are assigned by human users (e.g., marking emails as spam or not-spam). An attacker can send arbitrary emails but cannot guarantee that users will label them with the attacker's desired label. Similarly, in medical diagnosis, labels come from expert pathologists who would flag clearly mislabelled samples. In such settings, the attacker would need to craft data that is simultaneously adversarial in feature space *and* plausible enough to fool the labelling oracle — a much harder constraint not addressed by this method.

**Paper reference:**  
Section 4 (Conclusions): *"an important practical limitation of the proposed method is the assumption that the attacker controls the labels of the injected points. Such assumptions may not hold when the labels are only assigned by trusted sources such as humans."* Also Section 2.1: *"The choice of the attack point's label, $y_c$, is arbitrary but fixed."*

---
## Assumption 3: The SVM Solution Remains Structurally Stable Under Small Perturbations (Adiabatic Condition)

**Assumption:**  
When the attack point $x_c$ is moved by a small step $t \cdot u$, the partition of training points into margin support vectors ($S$, where $0 < \alpha_i < C$), error support vectors ($E$, where $\alpha_i = C$), and reserve points ($R$, where $\alpha_i = 0$) does not change. In other words, the set membership $S$, $E$, $R$ is locally constant around the current SVM solution.

**Why the method needs it:**  
The closed-form gradient in Eqs. (6)–(9) is derived by differentiating the KKT equality conditions that hold specifically for margin support vectors in set $S$. The inverse matrix $Q_{ss}^{-1}$ in Eq. (9) is defined only over the *current* set $S$. If a single gradient step causes a point to migrate between sets (e.g., a margin SV becomes a reserve point), the matrix dimensions and the entire gradient expression become invalid. The authors handle this by taking "many tiny gradient steps" (Section 4), essentially assuming that each step is small enough to preserve structural stability.

**Violation scenario:**  
This assumption is violated when the SVM has a very small margin or when the regularisation parameter $C$ is extreme (very large or very small). With $C \to \infty$, nearly all points become margin SVs and small perturbations easily push points across the $\alpha_i = C$ boundary. Similarly, if the data lies close to the decision boundary, even a small change in $x_c$ can cause multiple points to switch between $S$, $E$, and $R$. In such regimes, the fixed step size must be made extremely small, making the attack slow or impractical, and the gradient may exhibit discontinuities that prevent smooth convergence.

**Paper reference:**  
Section 2.1, Equations (4)–(5) define the KKT partition; Section 2.3 and Algorithm 1 use a fixed small step size $t$ to maintain this partition. Section 4 (Conclusions): *"The first [improvement] would be to address our optimization method's restriction to small changes in order to maintain the SVM's structural constraints. We solved this by taking many tiny gradient steps."*